In [1]:
import os
import re
import json
import urllib3
from time import sleep
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

urllib3.disable_warnings()

BASE_URL    = "https://sinaica.inecc.gob.mx/pags/datGrafs.php"
STATION_URL = "https://sinaica.inecc.gob.mx/pags/datosRed.php"

PARAMS      = ["PM2.5", "PM10", "O3", "NO2", "SO2", "CO"]
START       = pd.Timestamp("2022-01-01")
END         = pd.Timestamp("2025-12-31")
OUT_DIR     = "data/sinaica/"
MAX_WORKERS = 16


In [2]:
def get_stations() -> pd.DataFrame:
    r = requests.get(STATION_URL, verify=False, timeout=30)
    soup = BeautifulSoup(r.text, "html.parser")
    select = soup.find("select", {"id": "selPickHeadEst"})
    return pd.DataFrame([
        {
            "station_id":   opt["value"],
            "station_name": opt.text.strip(),
            "raw_tokens":   opt.get("data-tokens", ""),
        }
        for opt in select.find_all("option")
        if opt.get("value", "").strip()
    ])


def get_year(session, station_id, parameter, year_start) -> pd.DataFrame:
    ts    = pd.Timestamp(year_start)
    days  = 366 if ts.is_leap_year else 365
    payload = {
        "estacionId": station_id,
        "param":      parameter,
        "fechaIni":   ts.strftime("%Y-%m-%d"),
        "rango":      days,
        "tipoDatos":  "",
    }
    r = session.post(
        BASE_URL, data=payload,
        headers={"Referer": "https://sinaica.inecc.gob.mx/"},
        verify=False, timeout=30,
    )
    r.raise_for_status()

    match = re.search(r"var dat\s*=\s*(\[.*?\]);", r.text, re.DOTALL)
    if not match:
        return pd.DataFrame()
    data = json.loads(match.group(1))
    if not data:
        return pd.DataFrame()

    df = pd.DataFrame(data)
    df.columns = ["row_id", "date", "hour", "value", "extra", "valid"]
    df["datetime"] = (
        pd.to_datetime(df["date"])
        + pd.to_timedelta(pd.to_numeric(df["hour"], errors="coerce"), unit="h")
    )
    df["value"]      = pd.to_numeric(df["value"], errors="coerce")
    df["valid"]      = pd.to_numeric(df["valid"], errors="coerce")
    df["station_id"] = station_id
    df["pollutant"]  = parameter
    return df[["datetime", "station_id", "pollutant", "value", "valid"]]


def scrape_station(station_id, station_name) -> str:
    out_path = os.path.join(OUT_DIR, f"station_{station_id}.csv")
    if os.path.exists(out_path):
        return f"skip:{station_id}"

    session = requests.Session()
    all_dfs = []

    for param in PARAMS:
        current  = START.replace(month=1, day=1)
        end_year = END.replace(month=1, day=1)
        while current <= end_year:
            try:
                df = get_year(session, station_id, param, current)
                if not df.empty:
                    all_dfs.append(df)
            except Exception:
                pass
            current += pd.offsets.YearBegin(1)
            sleep(0.3)

    if not all_dfs:
        return f"empty:{station_id}"

    combined = pd.concat(all_dfs, ignore_index=True)
    combined = combined.dropna(subset=["value"])
    combined = combined[combined["valid"] == 1]
    combined.to_csv(out_path, index=False)
    return f"save:{station_id}:{len(combined)}"


In [3]:
# ── Fetch & save station list ─────────────────────────────────────────────────
df_stations = get_stations()
df_stations.to_csv(os.path.join(OUT_DIR, "sinaica_stations.csv"), index=False)
print(f"Found {len(df_stations)} stations")
df_stations.head()


Found 181 stations


,station_id,station_name,raw_tokens
0,356,Presidencia Municipal,Guanajuato GTO ABA Abasolo Presidencia Municip...
1,31,CBTIS,Aguascalientes AGS AGS Aguascalientes CBTIS CBT
2,33,Centro,Aguascalientes AGS AGS Aguascalientes Centro CEN
3,303,Instituto Educativo,Aguascalientes AGS AGS Aguascalientes Institut...
4,32,Secretaría de Medio Ambiente,Aguascalientes AGS AGS Aguascalientes Secretar...


In [6]:
# ── Test: one station, all pollutants ────────────────────────────────────────
# Change the index or set station_id/name manually
test_row      = df_stations.iloc[59]
test_id       = test_row["station_id"]
test_name     = test_row["station_name"]

print(f"Testing station {test_id} — {test_name}")
result = scrape_station(test_id, test_name)
print(f"Result: {result}")

out_path = os.path.join(OUT_DIR, f"station_{test_id}.csv")
if os.path.exists(out_path):
    df_test = pd.read_csv(out_path, parse_dates=["datetime"])
    print(f"\n{len(df_test)} rows | pollutants: {df_test['pollutant'].unique()}")
    display(df_test.head(10))


Testing station 495 — Mineral de la Reforma
Result: empty:495


In [5]:
df_test

,datetime,station_id,pollutant,value,valid
0,2022-01-01 01:00:00,356,PM2.5,22.517241,1
1,2022-01-01 02:00:00,356,PM2.5,38.192982,1
2,2022-01-01 03:00:00,356,PM2.5,70.491525,1
3,2022-01-01 04:00:00,356,PM2.5,56.000000,1
4,2022-01-01 05:00:00,356,PM2.5,54.711864,1
...,...,...,...,...,...
4704,2022-12-31 19:00:00,356,PM2.5,14.448276,1
4705,2022-12-31 20:00:00,356,PM2.5,26.762712,1
4706,2022-12-31 21:00:00,356,PM2.5,19.728814,1
4707,2022-12-31 22:00:00,356,PM2.5,12.000000,1


In [ ]:
# ── Scrape all stations (threaded) ───────────────────────────────────────────
os.makedirs(OUT_DIR, exist_ok=True)
saved, skipped, empty = 0, 0, 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(scrape_station, row["station_id"], row["station_name"]): row["station_id"]
        for _, row in df_stations.iterrows()
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc="Stations"):
        try:
            result = future.result()
            if result.startswith("save"):
                saved += 1
            elif result.startswith("skip"):
                skipped += 1
            else:
                empty += 1
        except Exception as e:
            print(f"Error on station {futures[future]}: {e}")
            empty += 1

print(f"\nDone. Saved: {saved} | Skipped (existing): {skipped} | No data: {empty}")
